<a href="https://colab.research.google.com/github/sadi-17/Mainshock-Discrimination-using-ML/blob/main/file_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import io
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
CONFIG = {
    "comcat": {
        "base":  "https://earthquake.usgs.gov/fdsnws/event/1/query",
        "count": "https://earthquake.usgs.gov/fdsnws/event/1/count",
        "start_year": 1930,
        "minmagnitude": 4.0,
        "extra_params": {},                      # global, no bbox
        "out": "comcat_global_m4.parquet",
    },
    "scedc": {
        "base":  "https://service.scedc.caltech.edu/fdsnws/event/1/query",
        "count": "https://service.scedc.caltech.edu/fdsnws/event/1/count",
        "start_year": 1981,
        "minmagnitude": 2.5,
        "extra_params": {},                      # SCEDC is regional already
        "out": "scedc_socal_m25.parquet",
    },
}

END = datetime.now(timezone.utc).replace(tzinfo=None)   # pull up to "now"
CHECKPOINT_DIR = Path("./chunks")                        # per-year caches
CHECKPOINT_DIR.mkdir(exist_ok=True)
CAP = 19000            # stay under the 20k hard limit with margin
POLITE_SLEEP = 0.5     # seconds between requests
MAX_RETRIES = 4

---------------------------------------------------------------------------
def _get(url, params, as_text=False):
    """GET with retries and exponential backoff."""
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, params=params, timeout=180)
            if r.status_code == 200:
                return r.text if as_text else r
            if r.status_code == 204:            # no content = zero events
                return "" if as_text else r
            print(f"  HTTP {r.status_code}, retry {attempt+1}")
        except requests.RequestException as e:
            print(f"  {type(e).__name__}, retry {attempt+1}")
        time.sleep(2 ** attempt * 2)
    raise RuntimeError(f"Failed after {MAX_RETRIES} retries: {params}")


def count_events(cfg, start, end):
    params = {
        "starttime": start.isoformat(), "endtime": end.isoformat(),
        "minmagnitude": cfg["minmagnitude"], **cfg["extra_params"],
    }
    txt = _get(cfg["count"], params, as_text=True).strip()
    return int(txt) if txt else 0


def fetch_window(cfg, start, end, depth=0):
    """Return list of DataFrames for [start, end); split recursively if > CAP."""
    n = count_events(cfg, start, end)
    if n == 0:
        return []
    if n > CAP:
        mid = start + (end - start) / 2
        return (fetch_window(cfg, start, mid, depth + 1)
                + fetch_window(cfg, mid, end, depth + 1))
    params = {
        "format": "csv", "orderby": "time-asc",
        "starttime": start.isoformat(), "endtime": end.isoformat(),
        "minmagnitude": cfg["minmagnitude"], **cfg["extra_params"],
    }
    txt = _get(cfg["base"], params, as_text=True)
    time.sleep(POLITE_SLEEP)
    if not txt.strip():
        return []
    df = pd.read_csv(io.StringIO(txt))
    print(f"  {'  '*depth}{start.date()} -> {end.date()}: {len(df):>6} events")
    return [df]


def fetch_year(cfg, year, name):
    """Download one calendar year, with on-disk checkpoint."""
    ckpt = CHECKPOINT_DIR / f"{name}_{year}.parquet"
    if ckpt.exists():
        return pd.read_parquet(ckpt)
    start = datetime(year, 1, 1)
    end = min(datetime(year + 1, 1, 1), END)
    frames = fetch_window(cfg, start, end)
    df = (pd.concat(frames, ignore_index=True) if frames
          else pd.DataFrame())
    df.to_parquet(ckpt, index=False)
    return df


# ----------------------------------------------------------------------------
# Cleaning + magnitude homogenization

MW_FAMILY = {"mw", "mww", "mwc", "mwb", "mwr", "mi", "mwp"}
ML_FAMILY = {"ml", "mlr", "mlg", "ml(texnet)", "mb_lg", "mblg", "md", "mh"}
MB_FAMILY = {"mb", "mb1", "mbmle"}
MS_FAMILY = {"ms", "ms_20", "msz"}


def to_mw(mag, mtype):
    t = str(mtype).lower()
    if t in MW_FAMILY:
        return mag, "direct"
    if t in MB_FAMILY:
        return 0.85 * mag + 0.33, "mb"
    if t in ML_FAMILY:
        return 0.85 * mag + 0.15, "ml"
    if t in MS_FAMILY:
        return 0.67 * mag + 2.07, "ms"
    return mag, "unconverted"          # kept + flagged (ablation later)


def clean(df):
    n0 = len(df)
    # tectonic earthquakes only (drops quarry blasts, explosions, etc.)
    if "type" in df.columns:
        df = df[df["type"] == "earthquake"]
    # essentials present
    df = df.dropna(subset=["time", "latitude", "longitude", "mag"])
    # nst is >90% missing in ComCat -> drop rather than let MICE hallucinate
    df = df.drop(columns=[c for c in ["nst"] if c in df.columns])
    # authoritative dedup on event id
    df = df.drop_duplicates(subset="id", keep="first")
    # parse time, sort chronologically
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").reset_index(drop=True)
    # magnitude homogenization
    conv = df.apply(lambda r: to_mw(r["mag"], r.get("magType")), axis=1)
    df["mw"] = [c[0] for c in conv]
    df["mw_source"] = [c[1] for c in conv]
    print(f"  cleaned: {n0} -> {len(df)} rows "
          f"({(df['mw_source']=='unconverted').sum()} unconverted magTypes)")
    return df


# ----------------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------------
def build(name):
    cfg = CONFIG[name]
    print(f"=== {name.upper()} : {cfg['start_year']}–{END.year}, "
          f"M>={cfg['minmagnitude']} ===")
    years = range(cfg["start_year"], END.year + 1)
    frames = [fetch_year(cfg, y, name) for y in years]
    raw = pd.concat([f for f in frames if len(f)], ignore_index=True)
    print(f"  raw events: {len(raw)}")
    cat = clean(raw)

    # snapshot metadata for reproducibility / Zenodo release
    meta = {
        "source": name,
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "start_year": cfg["start_year"],
        "end_utc": END.isoformat(),
        "minmagnitude": cfg["minmagnitude"],
        "n_events": int(len(cat)),
        "credit": ("U.S. Geological Survey ANSS ComCat" if name == "comcat"
                   else "SCEDC / Caltech-USGS SCSN, doi:10.7909/C3WD3xH1"),
        "note": "Derived data product; cite original network in publications.",
    }
    cat.attrs.update(meta)
    cat.to_parquet(cfg["out"], index=False)
    Path(cfg["out"]).with_suffix(".meta.json").write_text(json.dumps(meta, indent=2))
    print(f"  saved -> {cfg['out']}  (+ .meta.json)\n")
    return cat


if __name__ == "__main__":
    comcat = build("comcat")
    scedc = build("scedc")

    # ------------------------- quick sanity report -------------------------
    for nm, cat in [("ComCat", comcat), ("SCEDC", scedc)]:
        print(f"--- {nm} sanity ---")
        print("span:", cat["time"].min(), "->", cat["time"].max())
        print("events:", len(cat))
        print("magType mix:\n", cat["magType"].value_counts().head(8))
        print("events/decade:\n",
              cat.groupby(cat["time"].dt.year // 10 * 10).size(), "\n")

    # Ridgecrest ground-truth check (must contain M6.4 Jul-4 and M7.1 Jul-6 2019)
    rc = comcat[(comcat["time"].between("2019-07-04", "2019-07-07"))
                & (comcat["mw"] >= 6.0)]
    print("Ridgecrest check:\n", rc[["time", "mag", "magType", "mw", "place"]])

=== COMCAT : 1930–2026, M>=4.0 ===
  1930-01-01 -> 1931-01-01:    296 events
  1931-01-01 -> 1932-01-01:    303 events
  1932-01-01 -> 1933-01-01:    255 events
  1933-01-01 -> 1934-01-01:    383 events
  1934-01-01 -> 1935-01-01:    311 events
  1935-01-01 -> 1936-01-01:    281 events
  1936-01-01 -> 1937-01-01:    291 events
  1937-01-01 -> 1938-01-01:    255 events
  1938-01-01 -> 1939-01-01:    347 events
  1939-01-01 -> 1940-01-01:    256 events
  1940-01-01 -> 1941-01-01:    274 events
  1941-01-01 -> 1942-01-01:    225 events
  1942-01-01 -> 1943-01-01:    200 events
  1943-01-01 -> 1944-01-01:    178 events
  1944-01-01 -> 1945-01-01:    138 events
  1945-01-01 -> 1946-01-01:    140 events
  1946-01-01 -> 1947-01-01:    178 events
  1947-01-01 -> 1948-01-01:    180 events
  1948-01-01 -> 1949-01-01:    208 events
  1949-01-01 -> 1950-01-01:    165 events
  1950-01-01 -> 1951-01-01:    272 events
  1951-01-01 -> 1952-01-01:    284 events
  1952-01-01 -> 1953-01-01:    522 events

RuntimeError: Failed after 4 retries: {'starttime': '1981-01-01T00:00:00', 'endtime': '1982-01-01T00:00:00', 'minmagnitude': 2.5}

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

comcat = pd.read_parquet("comcat_global_m4.parquet")

BOX = dict(min_lat=20.0, max_lat=32.0, min_lon=85.0, max_lon=98.0)

sub = comcat[
    comcat["latitude"].between(BOX["min_lat"], BOX["max_lat"])
    & comcat["longitude"].between(BOX["min_lon"], BOX["max_lon"])
].reset_index(drop=True)

OUT = "comcat_himalaya_m4.parquet"
sub.to_parquet(OUT, index=False)

meta = {
    "source": "comcat_subset_himalaya",
    "bbox": BOX,
    "n_events": int(len(sub)),
    "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    "note": ("Bengal Basin / Himalayan arc subset of the global ComCat "
             "pull; derived product, credit USGS ANSS ComCat."),
}
Path(OUT).with_suffix(".meta.json").write_text(json.dumps(meta, indent=2))

print(f"Himalaya subset: {len(sub)} events -> {OUT}")
print(sub["time"].min(), "->", sub["time"].max())
print(sub["mw"].describe())

Himalaya subset: 4688 events -> comcat_himalaya_m4.parquet
1930-07-02 21:03:43.070000+00:00 -> 2026-06-22 15:28:52.406000+00:00
count    4688.000000
mean        4.313455
std         0.580573
min         3.635000
25%         3.985000
50%         4.155000
75%         4.410000
max         8.600000
Name: mw, dtype: float64


In [ ]:

import io
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

# ============================================================================
# 0. SETTINGS
# ============================================================================
END = datetime.now(timezone.utc).replace(tzinfo=None)
CHECKPOINT_DIR = Path("./chunks")          # per-year download cache
CHECKPOINT_DIR.mkdir(exist_ok=True)
ROW_CAP = 19000
POLITE_SLEEP = 0.5
MAX_RETRIES = 4

# --- Google Drive persistence: everything (checkpoints + final Parquets)
#     lives in MyDrive/quake_project, so Colab resets cost nothing. ---
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/quake_project', exist_ok=True)
os.chdir('/content/drive/MyDrive/quake_project')
CHECKPOINT_DIR = Path('./chunks')
CHECKPOINT_DIR.mkdir(exist_ok=True)

COMCAT_QUERY = "https://earthquake.usgs.gov/fdsnws/event/1/query"
COMCAT_COUNT = "https://earthquake.usgs.gov/fdsnws/event/1/count"
SCEDC_QUERY = "https://service.scedc.caltech.edu/fdsnws/event/1/query"

HIMALAYA_BOX = dict(min_lat=20.0, max_lat=32.0, min_lon=85.0, max_lon=98.0)


# ============================================================================
# 1. SHARED HELPERS
# ============================================================================
def _request(url, params):
    """GET with retries/backoff. Returns the Response, or None if all fail.
    Status 400/413 is returned immediately (caller splits the window)."""
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, params=params, timeout=180)
            if r.status_code in (200, 204, 400, 413):
                return r
            print(f"  HTTP {r.status_code}, retry {attempt + 1}")
        except requests.RequestException as e:
            print(f"  {type(e).__name__}, retry {attempt + 1}")
        time.sleep(2 ** attempt * 2)
    return None


def fetch_window(query_url, minmag, start, end, count_url=None, depth=0):
    """Fetch events in [start, end); recursively split when over the cap.
    Uses the /count endpoint when available (ComCat); otherwise splits
    reactively on server refusal (SCEDC has no /count)."""
    if (end - start).total_seconds() < 6 * 3600:
        raise RuntimeError(f"Unsplittable window {start}–{end}")

    def split():
        mid = start + (end - start) / 2
        return (fetch_window(query_url, minmag, start, mid, count_url, depth + 1)
                + fetch_window(query_url, minmag, mid, end, count_url, depth + 1))

    base_params = {
        "starttime": start.isoformat(), "endtime": end.isoformat(),
        "minmagnitude": minmag,
    }

    # proactive split when a count endpoint exists
    if count_url:
        r = _request(count_url, base_params)
        if r is None:
            raise RuntimeError(f"count failed for {start}–{end}")
        n = int(r.text.strip()) if r.text.strip() else 0
        if n == 0:
            return []
        if n > ROW_CAP:
            return split()

    r = _request(query_url,
                 {**base_params, "format": "csv", "orderby": "time-asc"})
    if r is None:
        raise RuntimeError(f"query failed for {start}–{end}")
    if r.status_code in (400, 413):      # too many events -> reactive split
        return split()
    if r.status_code == 204 or not r.text.strip():
        return []

    df = pd.read_csv(io.StringIO(r.text))
    if len(df) >= ROW_CAP:               # possible truncation -> split
        return split()
    print(f"  {'  ' * depth}{start.date()} -> {end.date()}: {len(df):>6} events")
    time.sleep(POLITE_SLEEP)
    return [df]


def fetch_catalog(name, query_url, minmag, start_year, count_url=None):
    """Download a full catalog year by year with on-disk checkpoints."""
    frames = []
    for year in range(start_year, END.year + 1):
        ckpt = CHECKPOINT_DIR / f"{name}_{year}.parquet"
        if ckpt.exists():
            frames.append(pd.read_parquet(ckpt))
            continue
        start = datetime(year, 1, 1)
        end = min(datetime(year + 1, 1, 1), END)
        year_frames = fetch_window(query_url, minmag, start, end, count_url)
        df = (pd.concat(year_frames, ignore_index=True)
              if year_frames else pd.DataFrame())
        df.to_parquet(ckpt, index=False)
        frames.append(df)
    return pd.concat([f for f in frames if len(f)], ignore_index=True)


# ============================================================================
# 2. CLEANING + MAGNITUDE HOMOGENIZATION
# ============================================================================
MW_FAMILY = {"mw", "mww", "mwc", "mwb", "mwr", "mi", "mwp"}
ML_FAMILY = {"ml", "mlr", "mlg", "ml(texnet)", "mb_lg", "mblg", "md", "mh"}
MB_FAMILY = {"mb", "mb1", "mbmle"}
MS_FAMILY = {"ms", "ms_20", "msz"}


def to_mw(mag, mtype):
    t = str(mtype).lower()
    if t in MW_FAMILY: return mag, "direct"
    if t in MB_FAMILY: return 0.85 * mag + 0.33, "mb"
    if t in ML_FAMILY: return 0.85 * mag + 0.15, "ml"
    if t in MS_FAMILY: return 0.67 * mag + 2.07, "ms"
    return mag, "unconverted"            # kept + flagged for the ablation


def clean(df):
    n0 = len(df)
    if "type" in df.columns:
        df = df[df["type"].astype(str).str.lower()
                .isin(["earthquake", "eq"])]
    df = df.dropna(subset=["time", "latitude", "longitude", "mag"])
    df = df.drop(columns=["nst"], errors="ignore")   # >90% missing
    if "id" in df.columns:
        df = df.drop_duplicates(subset="id", keep="first")
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").reset_index(drop=True)
    conv = df.apply(lambda r: to_mw(r["mag"], r.get("magType")), axis=1)
    df["mw"] = [c[0] for c in conv]
    df["mw_source"] = [c[1] for c in conv]
    print(f"  cleaned: {n0} -> {len(df)} rows "
          f"({(df['mw_source'] == 'unconverted').sum()} unconverted)")
    return df


def save(df, out, meta):
    df.to_parquet(out, index=False)
    Path(out).with_suffix(".meta.json").write_text(json.dumps(meta, indent=2))
    print(f"  saved -> {out}\n")


# ============================================================================
# 3. BUILD ALL THREE DATASETS
# ============================================================================
def main():
    # ---------- Dataset 1: ComCat global ----------
    if Path("comcat_global_m4.parquet").exists():
        print("[1/3] comcat_global_m4.parquet already exists — skipping.")
        comcat = pd.read_parquet("comcat_global_m4.parquet")
    else:
        print("[1/3] ComCat global, 1930–now, M>=4.0")
        raw = fetch_catalog("comcat", COMCAT_QUERY, 4.0, 1930,
                            count_url=COMCAT_COUNT)
        print(f"  raw events: {len(raw)}")
        comcat = clean(raw)
        save(comcat, "comcat_global_m4.parquet", {
            "source": "comcat", "minmagnitude": 4.0, "start_year": 1930,
            "end_utc": END.isoformat(), "n_events": int(len(comcat)),
            "retrieved_utc": datetime.now(timezone.utc).isoformat(),
            "credit": "U.S. Geological Survey ANSS ComCat",
            "note": "Derived data product; cite USGS in publications.",
        })

    # ---------- Dataset 2: SCEDC Southern California ----------
    if Path("scedc_socal_m25.parquet").exists():
        print("[2/3] scedc_socal_m25.parquet already exists — skipping.")
        scedc = pd.read_parquet("scedc_socal_m25.parquet")
    else:
        print("[2/3] SCEDC Southern California, 1981–now, M>=2.5 "
              "(no /count endpoint — splits reactively)")
        raw = fetch_catalog("scedc", SCEDC_QUERY, 2.5, 1981, count_url=None)
        print(f"  raw events: {len(raw)}")
        scedc = clean(raw)
        save(scedc, "scedc_socal_m25.parquet", {
            "source": "scedc", "minmagnitude": 2.5, "start_year": 1981,
            "end_utc": END.isoformat(), "n_events": int(len(scedc)),
            "retrieved_utc": datetime.now(timezone.utc).isoformat(),
            "credit": "SCEDC / Caltech-USGS SCSN, doi:10.7909/C3WD3xH1",
            "note": "Derived data product; cite SCEDC in publications.",
        })

    # ---------- Dataset 3: Himalaya subset (filter only, no download) ----------
    print("[3/3] Bengal/Himalaya subset of ComCat")
    b = HIMALAYA_BOX
    him = comcat[
        comcat["latitude"].between(b["min_lat"], b["max_lat"])
        & comcat["longitude"].between(b["min_lon"], b["max_lon"])
    ].reset_index(drop=True)
    save(him, "comcat_himalaya_m4.parquet", {
        "source": "comcat_subset_himalaya", "bbox": b,
        "n_events": int(len(him)),
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "note": "Bengal Basin / Himalayan arc subset; credit USGS ComCat.",
    })

    # ---------- Sanity report ----------
    for nm, cat in [("ComCat", comcat), ("SCEDC", scedc), ("Himalaya", him)]:
        print(f"--- {nm} ---")
        print("  events:", len(cat),
              "| span:", cat["time"].min().date(), "->", cat["time"].max().date())
    rc = comcat[(comcat["time"].between("2019-07-04", "2019-07-07"))
                & (comcat["mw"] >= 6.0)]
    print("\nRidgecrest check (expect M6.4 Jul-4 and M7.1 Jul-6, 2019):")
    print(rc[["time", "mag", "magType", "mw", "place"]].to_string(index=False))


if __name__ == "__main__":
    main()

Mounted at /content/drive
[1/3] ComCat global, 1930–now, M>=4.0
  1930-01-01 -> 1931-01-01:    296 events
  1931-01-01 -> 1932-01-01:    303 events
  1932-01-01 -> 1933-01-01:    255 events
  1933-01-01 -> 1934-01-01:    383 events
  1934-01-01 -> 1935-01-01:    311 events
  1935-01-01 -> 1936-01-01:    281 events
  1936-01-01 -> 1937-01-01:    291 events
  1937-01-01 -> 1938-01-01:    255 events
  1938-01-01 -> 1939-01-01:    347 events
  1939-01-01 -> 1940-01-01:    256 events
  1940-01-01 -> 1941-01-01:    274 events
  1941-01-01 -> 1942-01-01:    225 events
  1942-01-01 -> 1943-01-01:    200 events
  1943-01-01 -> 1944-01-01:    178 events
  1944-01-01 -> 1945-01-01:    138 events
  1945-01-01 -> 1946-01-01:    140 events
  1946-01-01 -> 1947-01-01:    178 events
  1947-01-01 -> 1948-01-01:    180 events
  1948-01-01 -> 1949-01-01:    208 events
  1949-01-01 -> 1950-01-01:    165 events
  1950-01-01 -> 1951-01-01:    272 events
  1951-01-01 -> 1952-01-01:    284 events
  1952-01-01

RuntimeError: Unsplittable window 1981-01-01 00:00:00–1981-01-01 04:16:38.437500

In [ ]:


import io
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

# ============================================================================
# 0. SETTINGS
# ============================================================================
END = datetime.now(timezone.utc).replace(tzinfo=None)
CHECKPOINT_DIR = Path("./chunks")          # per-year download cache
CHECKPOINT_DIR.mkdir(exist_ok=True)
ROW_CAP = 19000
POLITE_SLEEP = 0.5
MAX_RETRIES = 4

# --- Google Drive persistence: everything (checkpoints + final Parquets)
#     lives in MyDrive/quake_project, so Colab resets cost nothing. ---
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/quake_project', exist_ok=True)
os.chdir('/content/drive/MyDrive/quake_project')
CHECKPOINT_DIR = Path('./chunks')
CHECKPOINT_DIR.mkdir(exist_ok=True)

COMCAT_QUERY = "https://earthquake.usgs.gov/fdsnws/event/1/query"
COMCAT_COUNT = "https://earthquake.usgs.gov/fdsnws/event/1/count"
SCEDC_QUERY = "https://service.scedc.caltech.edu/fdsnws/event/1/query"

HIMALAYA_BOX = dict(min_lat=20.0, max_lat=32.0, min_lon=85.0, max_lon=98.0)


# ============================================================================
# 1. SHARED HELPERS
# ============================================================================
def _request(url, params):
    """GET with retries/backoff. Returns the Response, or None if all fail.
    Status 400/413 is returned immediately (caller splits the window)."""
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, params=params, timeout=180)
            if r.status_code in (200, 204, 400, 413):
                return r
            print(f"  HTTP {r.status_code}, retry {attempt + 1}")
        except requests.RequestException as e:
            print(f"  {type(e).__name__}, retry {attempt + 1}")
        time.sleep(2 ** attempt * 2)
    return None


FDSN_TEXT_COLS = ["id", "time", "latitude", "longitude", "depth", "author",
                  "catalog", "contributor", "contributor_id", "magType",
                  "mag", "mag_author", "place"]


def parse_fdsn_text(txt):
    """Parse FDSN-standard pipe-delimited text into ComCat-like columns."""
    df = pd.read_csv(io.StringIO(txt), sep="|", comment=None, header=0,
                     names=FDSN_TEXT_COLS, dtype=str)
    for col in ["latitude", "longitude", "depth", "mag"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["type"] = "earthquake"        # SCEDC text output is earthquakes only
    return df


def fetch_window(query_url, minmag, start, end, count_url=None, depth=0,
                 fmt="csv"):
    """Fetch events in [start, end); recursively split when over the cap.
    Uses the /count endpoint when available (ComCat); otherwise splits
    reactively on server refusal (SCEDC has no /count).
    fmt: 'csv' for ComCat, 'text' (FDSN standard) for SCEDC."""
    if (end - start).total_seconds() < 6 * 3600:
        raise RuntimeError(f"Unsplittable window {start}–{end}")

    def split():
        mid = start + (end - start) / 2
        return (fetch_window(query_url, minmag, start, mid, count_url,
                             depth + 1, fmt)
                + fetch_window(query_url, minmag, mid, end, count_url,
                               depth + 1, fmt))

    base_params = {
        "starttime": start.isoformat(), "endtime": end.isoformat(),
        "minmagnitude": minmag,
    }

    # proactive split when a count endpoint exists
    if count_url:
        r = _request(count_url, base_params)
        if r is None:
            raise RuntimeError(f"count failed for {start}–{end}")
        n = int(r.text.strip()) if r.text.strip() else 0
        if n == 0:
            return []
        if n > ROW_CAP:
            return split()

    r = _request(query_url,
                 {**base_params, "format": fmt, "orderby": "time-asc"})
    if r is None:
        raise RuntimeError(f"query failed for {start}–{end}")
    if r.status_code in (400, 413):      # too many events -> reactive split
        return split()
    if r.status_code == 204 or not r.text.strip():
        return []

    df = (parse_fdsn_text(r.text) if fmt == "text"
          else pd.read_csv(io.StringIO(r.text)))
    if len(df) >= ROW_CAP:               # possible truncation -> split
        return split()
    print(f"  {'  ' * depth}{start.date()} -> {end.date()}: {len(df):>6} events")
    time.sleep(POLITE_SLEEP)
    return [df]


def fetch_catalog(name, query_url, minmag, start_year, count_url=None,
                  fmt="csv"):
    """Download a full catalog year by year with on-disk checkpoints."""
    frames = []
    for year in range(start_year, END.year + 1):
        ckpt = CHECKPOINT_DIR / f"{name}_{year}.parquet"
        if ckpt.exists():
            frames.append(pd.read_parquet(ckpt))
            continue
        start = datetime(year, 1, 1)
        end = min(datetime(year + 1, 1, 1), END)
        year_frames = fetch_window(query_url, minmag, start, end, count_url,
                                   fmt=fmt)
        df = (pd.concat(year_frames, ignore_index=True)
              if year_frames else pd.DataFrame())
        df.to_parquet(ckpt, index=False)
        frames.append(df)
    return pd.concat([f for f in frames if len(f)], ignore_index=True)


# ============================================================================
# 2. CLEANING + MAGNITUDE HOMOGENIZATION
# ============================================================================
MW_FAMILY = {"mw", "mww", "mwc", "mwb", "mwr", "mi", "mwp"}
ML_FAMILY = {"ml", "mlr", "mlg", "ml(texnet)", "mb_lg", "mblg", "md", "mh"}
MB_FAMILY = {"mb", "mb1", "mbmle"}
MS_FAMILY = {"ms", "ms_20", "msz"}


def to_mw(mag, mtype):
    t = str(mtype).lower()
    if t in MW_FAMILY: return mag, "direct"
    if t in MB_FAMILY: return 0.85 * mag + 0.33, "mb"
    if t in ML_FAMILY: return 0.85 * mag + 0.15, "ml"
    if t in MS_FAMILY: return 0.67 * mag + 2.07, "ms"
    return mag, "unconverted"            # kept + flagged for the ablation


def clean(df):
    n0 = len(df)
    if "type" in df.columns:
        df = df[df["type"].astype(str).str.lower()
                .isin(["earthquake", "eq"])]
    df = df.dropna(subset=["time", "latitude", "longitude", "mag"])
    df = df.drop(columns=["nst"], errors="ignore")   # >90% missing
    if "id" in df.columns:
        df = df.drop_duplicates(subset="id", keep="first")
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").reset_index(drop=True)
    conv = df.apply(lambda r: to_mw(r["mag"], r.get("magType")), axis=1)
    df["mw"] = [c[0] for c in conv]
    df["mw_source"] = [c[1] for c in conv]
    print(f"  cleaned: {n0} -> {len(df)} rows "
          f"({(df['mw_source'] == 'unconverted').sum()} unconverted)")
    return df


def save(df, out, meta):
    df.to_parquet(out, index=False)
    Path(out).with_suffix(".meta.json").write_text(json.dumps(meta, indent=2))
    print(f"  saved -> {out}\n")


# ============================================================================
# 3. BUILD ALL THREE DATASETS
# ============================================================================
def main():
    # ---------- Dataset 1: ComCat global ----------
    if Path("comcat_global_m4.parquet").exists():
        print("[1/3] comcat_global_m4.parquet already exists — skipping.")
        comcat = pd.read_parquet("comcat_global_m4.parquet")
    else:
        print("[1/3] ComCat global, 1930–now, M>=4.0")
        raw = fetch_catalog("comcat", COMCAT_QUERY, 4.0, 1930,
                            count_url=COMCAT_COUNT)
        print(f"  raw events: {len(raw)}")
        comcat = clean(raw)
        save(comcat, "comcat_global_m4.parquet", {
            "source": "comcat", "minmagnitude": 4.0, "start_year": 1930,
            "end_utc": END.isoformat(), "n_events": int(len(comcat)),
            "retrieved_utc": datetime.now(timezone.utc).isoformat(),
            "credit": "U.S. Geological Survey ANSS ComCat",
            "note": "Derived data product; cite USGS in publications.",
        })

    # ---------- Dataset 2: SCEDC Southern California ----------
    if Path("scedc_socal_m25.parquet").exists():
        print("[2/3] scedc_socal_m25.parquet already exists — skipping.")
        scedc = pd.read_parquet("scedc_socal_m25.parquet")
    else:
        print("[2/3] SCEDC Southern California, 1981–now, M>=2.5 "
              "(no /count endpoint — splits reactively)")
        raw = fetch_catalog("scedc", SCEDC_QUERY, 2.5, 1981,
                            count_url=None, fmt="text")
        print(f"  raw events: {len(raw)}")
        scedc = clean(raw)
        save(scedc, "scedc_socal_m25.parquet", {
            "source": "scedc", "minmagnitude": 2.5, "start_year": 1981,
            "end_utc": END.isoformat(), "n_events": int(len(scedc)),
            "retrieved_utc": datetime.now(timezone.utc).isoformat(),
            "credit": "SCEDC / Caltech-USGS SCSN, doi:10.7909/C3WD3xH1",
            "note": "Derived data product; cite SCEDC in publications.",
        })

    # ---------- Dataset 3: Himalaya subset (filter only, no download) ----------
    print("[3/3] Bengal/Himalaya subset of ComCat")
    b = HIMALAYA_BOX
    him = comcat[
        comcat["latitude"].between(b["min_lat"], b["max_lat"])
        & comcat["longitude"].between(b["min_lon"], b["max_lon"])
    ].reset_index(drop=True)
    save(him, "comcat_himalaya_m4.parquet", {
        "source": "comcat_subset_himalaya", "bbox": b,
        "n_events": int(len(him)),
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
        "note": "Bengal Basin / Himalayan arc subset; credit USGS ComCat.",
    })

    # ---------- Sanity report ----------
    for nm, cat in [("ComCat", comcat), ("SCEDC", scedc), ("Himalaya", him)]:
        print(f"--- {nm} ---")
        print("  events:", len(cat),
              "| span:", cat["time"].min().date(), "->", cat["time"].max().date())
    rc = comcat[(comcat["time"].between("2019-07-04", "2019-07-07"))
                & (comcat["mw"] >= 6.0)]
    print("\nRidgecrest check (expect M6.4 Jul-4 and M7.1 Jul-6, 2019):")
    print(rc[["time", "mag", "magType", "mw", "place"]].to_string(index=False))


if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[1/3] comcat_global_m4.parquet already exists — skipping.
[2/3] SCEDC Southern California, 1981–now, M>=2.5 (no /count endpoint — splits reactively)
  1981-01-01 -> 1982-01-01:   2290 events
  1982-01-01 -> 1983-01-01:   2985 events
  1983-01-01 -> 1984-01-01:   2936 events
  1984-01-01 -> 1985-01-01:   2571 events
  1985-01-01 -> 1986-01-01:   2544 events
  1986-01-01 -> 1987-01-01:   4169 events
  1987-01-01 -> 1988-01-01:   3487 events
  1988-01-01 -> 1989-01-01:   2649 events
  1989-01-01 -> 1990-01-01:   2261 events
  1990-01-01 -> 1991-01-01:   2203 events
  1991-01-01 -> 1992-01-01:   1973 events
  1992-01-01 -> 1993-01-01:   7004 events
  1993-01-01 -> 1994-01-01:   1515 events
  1994-01-01 -> 1995-01-01:   2203 events
  1995-01-01 -> 1996-01-01:   1182 events
  1996-01-01 -> 1997-01-01:   1153 events
  1997-01-01 -> 1998-01-01:   1289 events
  1998-0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/quake_project')
print(sorted(os.listdir()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['chunks', 'comcat_global_m4.meta.json', 'comcat_global_m4.parquet', 'comcat_global_m4_features.parquet', 'comcat_global_m4_labeled.parquet', 'comcat_himalaya_m4.meta.json', 'comcat_himalaya_m4.parquet', 'comcat_himalaya_m4_features.parquet', 'comcat_himalaya_m4_labeled.parquet', 'feat_ckpt', 'label_ckpt', 'labeling_summary.csv', 'models', 'scedc_socal_m25.meta.json', 'scedc_socal_m25.parquet', 'scedc_socal_m25_features.parquet', 'scedc_socal_m25_labeled.parquet']


In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# ----------------------------------------------------------------------------
CATALOGS = [
    "comcat_global_m4.parquet",
    "scedc_socal_m25.parquet",
    "comcat_himalaya_m4.parquet",
]
RADII_KM = [25, 50, 75, 100]
WINDOWS_DAYS = [15, 30, 60, 90]
REFERENCE = (50, 30)

EARTH_R = 6371.0
BATCH = 200                              # small on purpose: RAM safety
LABEL_CKPT = Path("./label_ckpt")
LABEL_CKPT.mkdir(exist_ok=True)

ADAPT_R = lambda m: np.minimum(10 ** (0.1238 * m + 0.983), 100.0)   # km
ADAPT_T = lambda m: np.minimum(10 ** (0.5 * m - 0.547), 365.0)      # days


# ----------------------------------------------------------------------------
def to_unit_xyz(lat_deg, lon_deg):
    lat, lon = np.radians(lat_deg), np.radians(lon_deg)
    return np.column_stack([np.cos(lat) * np.cos(lon),
                            np.cos(lat) * np.sin(lon),
                            np.sin(lat)])


def chord_from_km(d_km):
    return 2.0 * np.sin(d_km / (2.0 * EARTH_R))


# ----------------------------------------------------------------------------
def label_fixed(times_s, mw, tree, xyz, R_km, T_days):
    n = len(mw)
    labels = np.ones(n, dtype=np.int8)
    T_s = T_days * 86400.0
    chord = chord_from_km(R_km)
    for lo in range(0, n, BATCH):
        hi = min(lo + BATCH, n)
        neigh_lists = tree.query_ball_point(xyz[lo:hi], r=chord)
        for k in range(hi - lo):
            i = lo + k
            nb = np.asarray(neigh_lists[k])
            neigh_lists[k] = None                    # free as we go
            nb = nb[nb != i]
            if nb.size == 0:
                continue
            nb = nb[np.abs(times_s[nb] - times_s[i]) <= T_s]
            if nb.size and (mw[nb] > mw[i]).any():
                labels[i] = 0
        del neigh_lists
        if (lo // BATCH) % 200 == 0:
            gc.collect()
    return labels


def label_adaptive(times_s, mw, tree, xyz):
    n = len(mw)
    labels = np.ones(n, dtype=np.int8)
    max_chord = chord_from_km(100.0)
    for lo in range(0, n, BATCH):
        hi = min(lo + BATCH, n)
        neigh_lists = tree.query_ball_point(xyz[lo:hi], r=max_chord)
        for k in range(hi - lo):
            i = lo + k
            nb = np.asarray(neigh_lists[k])
            neigh_lists[k] = None
            nb = nb[nb != i]
            if nb.size == 0:
                continue
            bigger = nb[mw[nb] > mw[i]]
            if bigger.size == 0:
                continue
            d_chord = np.linalg.norm(xyz[bigger] - xyz[i], axis=1)
            d_km = 2.0 * EARTH_R * np.arcsin(np.clip(d_chord / 2.0, 0, 1))
            dt_days = np.abs(times_s[bigger] - times_s[i]) / 86400.0
            if ((d_km <= ADAPT_R(mw[bigger]))
                    & (dt_days <= ADAPT_T(mw[bigger]))).any():
                labels[i] = 0
        del neigh_lists
        if (lo // BATCH) % 200 == 0:
            gc.collect()
    return labels


# ----------------------------------------------------------------------------
def all_configs():
    cfgs = [(f"label_R{R}_T{T}", ("fixed", R, T))
            for R in RADII_KM for T in WINDOWS_DAYS]
    cfgs.append(("label_adaptive", ("adaptive", None, None)))
    return cfgs


def process(path):
    name = Path(path).stem
    out_path = f"{name}_labeled.parquet"
    if Path(out_path).exists():
        print(f"[skip] {out_path} already exists")
        return name

    print(f"=== Labeling {name} ===")
    df = pd.read_parquet(path).sort_values("time").reset_index(drop=True)
    times_s = ((df["time"] - pd.Timestamp(0, tz="UTC"))
               .dt.total_seconds().to_numpy())
    mw = df["mw"].to_numpy(dtype=np.float64)
    xyz = to_unit_xyz(df["latitude"].to_numpy(), df["longitude"].to_numpy())
    tree = cKDTree(xyz)
    print(f"  {len(df)} events, KD-tree built")

    for col, (kind, R, T) in all_configs():
        ckpt = LABEL_CKPT / f"{name}__{col}.npy"
        if ckpt.exists():
            labels = np.load(ckpt)
            print(f"  {col}: loaded from checkpoint "
                  f"(rate = {labels.mean():.3f})")
        else:
            labels = (label_fixed(times_s, mw, tree, xyz, R, T)
                      if kind == "fixed"
                      else label_adaptive(times_s, mw, tree, xyz))
            np.save(ckpt, labels)
            print(f"  {col}: mainshock rate = {labels.mean():.3f}")
        df[col] = labels
        gc.collect()

    df.to_parquet(out_path, index=False)
    print(f"  saved -> {out_path}\n")
    del df, times_s, mw, xyz, tree
    gc.collect()
    return name


def summarize(names):
    ref_col = f"label_R{REFERENCE[0]}_T{REFERENCE[1]}"
    rows = []
    for name in names:
        cols = [c for c, _ in all_configs()]
        lab = pd.read_parquet(f"{name}_labeled.parquet",
                              columns=cols)          # labels only: tiny
        for col in cols:
            rows.append({
                "catalog": name, "config": col,
                "mainshock_rate": round(float(lab[col].mean()), 4),
                "flip_rate_vs_ref":
                    round(float((lab[col] != lab[ref_col]).mean()), 4),
            })
        del lab
        gc.collect()
    summary = pd.DataFrame(rows)
    summary.to_csv("labeling_summary.csv", index=False)
    print("=== Summary (saved to labeling_summary.csv) ===")
    print(summary.to_string(index=False))


if __name__ == "__main__":
    names = [process(p) for p in CATALOGS]
    summarize(names)

    ref_col = f"label_R{REFERENCE[0]}_T{REFERENCE[1]}"
    com = pd.read_parquet("comcat_global_m4_labeled.parquet",
                          columns=["time", "mw", "place",
                                   ref_col, "label_adaptive"])
    rc = com[(com["time"].between("2019-07-04", "2019-07-07"))
             & (com["mw"] >= 6.0)]
    print("\nRidgecrest label check (expect M6.4 -> 0, M7.1 -> 1):")
    print(rc.to_string(index=False))

[skip] comcat_global_m4_labeled.parquet already exists
=== Labeling scedc_socal_m25 ===
  81109 events, KD-tree built
  label_R25_T15: mainshock rate = 0.405
  label_R25_T30: mainshock rate = 0.354
  label_R25_T60: mainshock rate = 0.306
  label_R25_T90: mainshock rate = 0.281
  label_R50_T15: mainshock rate = 0.328
  label_R50_T30: mainshock rate = 0.276
  label_R50_T60: mainshock rate = 0.233
  label_R50_T90: mainshock rate = 0.212
  label_R75_T15: mainshock rate = 0.283
  label_R75_T30: mainshock rate = 0.236
  label_R75_T60: mainshock rate = 0.198
  label_R75_T90: mainshock rate = 0.179
  label_R100_T15: mainshock rate = 0.254
  label_R100_T30: mainshock rate = 0.211
  label_R100_T60: mainshock rate = 0.176
  label_R100_T90: mainshock rate = 0.159
  label_adaptive: mainshock rate = 0.334
  saved -> scedc_socal_m25_labeled.parquet

=== Labeling comcat_himalaya_m4 ===
  4688 events, KD-tree built
  label_R25_T15: mainshock rate = 0.784
  label_R25_T30: mainshock rate = 0.755
  label_

In [ ]:

import gc
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# ----------------------------------------------------------------------------
CATALOGS = [
    "comcat_global_m4_labeled.parquet",
    "scedc_socal_m25_labeled.parquet",
    "comcat_himalaya_m4_labeled.parquet",
]
FEAT_R_KM = 50.0          # feature neighborhood (fixed across label grid)
EARTH_R = 6371.0
BATCH = 200
GAP_CAP_DAYS = 3650.0     # cap "time since last local event"
EPS = 1e-9

FEAT_CKPT = Path("./feat_ckpt")
FEAT_CKPT.mkdir(exist_ok=True)

# station-quality / location covariates to carry through when present
CARRY = ["depth", "gap", "rms", "dmin", "horizontalError", "depthError",
         "magError", "magNst", "latitude", "longitude"]


def to_unit_xyz(lat_deg, lon_deg):
    lat, lon = np.radians(lat_deg), np.radians(lon_deg)
    return np.column_stack([np.cos(lat) * np.cos(lon),
                            np.cos(lat) * np.sin(lon),
                            np.sin(lat)])


def chord_from_km(d_km):
    return 2.0 * np.sin(d_km / (2.0 * EARTH_R))


# ----------------------------------------------------------------------------
def compute_features(times_s, mw, tree, xyz):
    """One pass over all events; returns dict of five feature arrays.
    For each event, queries 50-km neighbors once, then applies the
    different backward time masks (7 d, 30 d, 365 d) to that one list."""
    n = len(mw)
    f = {
        "local_count_7d": np.zeros(n, dtype=np.float32),
        "log_energy_30d": np.zeros(n, dtype=np.float32),
        "mag_diff_30d": np.zeros(n, dtype=np.float32),
        "time_gap_local": np.full(n, GAP_CAP_DAYS, dtype=np.float32),
        "seismicity_z": np.zeros(n, dtype=np.float32),
    }
    chord = chord_from_km(FEAT_R_KM)
    D7, D30, D365 = 7 * 86400.0, 30 * 86400.0, 365 * 86400.0

    for lo in range(0, n, BATCH):
        hi = min(lo + BATCH, n)
        neigh_lists = tree.query_ball_point(xyz[lo:hi], r=chord)
        for k in range(hi - lo):
            i = lo + k
            nb = np.asarray(neigh_lists[k])
            neigh_lists[k] = None
            nb = nb[nb != i]
            if nb.size == 0:
                f["mag_diff_30d"][i] = mw[i]          # vs nothing
                continue
            dt = times_s[i] - times_s[nb]             # >0 means "in the past"
            past = nb[dt > 0]
            dt = dt[dt > 0]
            if past.size == 0:
                f["mag_diff_30d"][i] = mw[i]
                continue

            # time since most recent local event
            f["time_gap_local"][i] = min(dt.min() / 86400.0, GAP_CAP_DAYS)

            m7 = dt <= D7
            m30 = dt <= D30
            m365 = dt <= D365

            x7 = float(m7.sum())
            f["local_count_7d"][i] = x7

            if m30.any():
                mw30 = mw[past[m30]]
                f["log_energy_30d"][i] = np.log10(
                    np.sum(10.0 ** (1.5 * mw30)) + EPS)
                f["mag_diff_30d"][i] = mw[i] - mw30.max()
            else:
                f["mag_diff_30d"][i] = mw[i]

            mu = (float(m365.sum()) / 365.0) * 7.0     # expected 7-day count
            f["seismicity_z"][i] = (x7 - mu) / np.sqrt(mu + EPS)
        del neigh_lists
        if (lo // BATCH) % 200 == 0:
            gc.collect()
    return f


# ----------------------------------------------------------------------------
def process(path):
    name = Path(path).stem.replace("_labeled", "")
    out_path = f"{name}_features.parquet"
    if Path(out_path).exists():
        print(f"[skip] {out_path} already exists")
        return

    print(f"=== Features: {name} ===")
    df = pd.read_parquet(path).sort_values("time").reset_index(drop=True)
    times_s = ((df["time"] - pd.Timestamp(0, tz="UTC"))
               .dt.total_seconds().to_numpy())
    mw = df["mw"].to_numpy(dtype=np.float64)
    xyz = to_unit_xyz(df["latitude"].to_numpy(), df["longitude"].to_numpy())
    tree = cKDTree(xyz)
    print(f"  {len(df)} events, KD-tree built")

    ckpt = FEAT_CKPT / f"{name}__features.npz"
    if ckpt.exists():
        print("  loaded features from checkpoint")
        feats = dict(np.load(ckpt))
    else:
        feats = compute_features(times_s, mw, tree, xyz)
        np.savez(ckpt, **feats)
        print("  features computed + checkpointed")

    for col, arr in feats.items():
        df[col] = arr

    keep = (["id", "time", "mw", "mw_source", "place"]
            + [c for c in CARRY if c in df.columns]
            + [c for c in df.columns if c.startswith("label")]
            + list(feats.keys()))
    keep = [c for c in keep if c in df.columns]
    df[keep].to_parquet(out_path, index=False)

    print(f"  saved -> {out_path}")
    print(df[list(feats.keys())].describe().T
          [["mean", "std", "min", "max"]].round(3).to_string(), "\n")
    del df, feats, times_s, mw, xyz, tree
    gc.collect()


if __name__ == "__main__":
    for p in CATALOGS:
        process(p)

    # Ridgecrest feature sanity: the M7.1 should show a strongly POSITIVE
    # mag_diff? No: its biggest prior 30-d neighbor is the M6.4 -> +0.7.
    # The M6.4's prior neighborhood is quiet -> large positive too. The
    # aftershocks that follow should show sharply NEGATIVE mag_diff.
    com = pd.read_parquet("comcat_global_m4_features.parquet",
                          columns=["time", "mw", "place", "mag_diff_30d",
                                   "local_count_7d", "time_gap_local"])
    rc = com[(com["time"].between("2019-07-04", "2019-07-08"))
             & (com["place"].str.contains("Ridgecrest", na=False))
             & (com["mw"] >= 5.0)]
    print("Ridgecrest feature check (M>=5 sequence events):")
    print(rc.to_string(index=False))

=== Features: comcat_global_m4 ===
  541817 events, KD-tree built
  features computed + checkpointed
  saved -> comcat_global_m4_features.parquet
                   mean      std    min       max
local_count_7d    5.884   23.051  0.000   586.000
log_energy_30d    4.296    4.080  0.000    14.255
mag_diff_30d      1.512    2.755 -5.370     9.200
time_gap_local  283.134  758.678  0.000  3650.000
seismicity_z      5.111   13.793 -4.106   162.312 

=== Features: scedc_socal_m25 ===
  81109 events, KD-tree built
  features computed + checkpointed
  saved -> scedc_socal_m25_features.parquet
                   mean      std    min       max
local_count_7d   86.076  248.515  0.000  2027.000
log_energy_30d    5.416    3.615  0.000    13.501
mag_diff_30d      0.063    3.094 -4.800     9.000
time_gap_local  322.576  925.394  0.000  3650.000
seismicity_z     21.522   52.243 -8.907   316.685 

=== Features: comcat_himalaya_m4 ===
  4688 events, KD-tree built
  features computed + checkpointed
  save

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/quake_project')
print(os.listdir())   # should show your parquet files

Mounted at /content/drive
['chunks', 'comcat_global_m4.parquet', 'comcat_global_m4.meta.json', 'scedc_socal_m25.parquet', 'scedc_socal_m25.meta.json', 'comcat_himalaya_m4.parquet', 'comcat_himalaya_m4.meta.json', 'comcat_global_m4_labeled.parquet', 'label_ckpt', 'scedc_socal_m25_labeled.parquet', 'comcat_himalaya_m4_labeled.parquet', 'labeling_summary.csv', 'feat_ckpt', 'comcat_global_m4_features.parquet', 'scedc_socal_m25_features.parquet', 'comcat_himalaya_m4_features.parquet', 'models', 'catboost_info']


In [ ]:
import pandas as pd
df = pd.read_parquet("scedc_socal_m25.parquet")
print(df.groupby(df["time"].dt.year).size().to_string())

time
1981    2289
1982    2984
1983    2935
1984    2570
1985    2543
1986    4168
1987    3486
1988    2648
1989    2260
1990    2202
1991    1972
1992    7003
1993    1514
1994    2202
1995    1181
1996    1152
1997    1288
1998    1423
1999    3162
2000    1337
2001    1316
2002    1017
2003     998
2004    1106
2005    1068
2006     891
2007     932
2008    1227
2009    1148
2010    4944
2011    1389
2012    1143
2013     939
2014    1008
2015     687
2016     783
2017     608
2018     634
2019    3403
2020    1252
2021     879
2022     676
2023     714
2024     881
2025     757
2026     390


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 12.6 MB/s eta 0:00:00


In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [ ]:
import gc
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
CATALOGS = [
    "comcat_global_m4_features.parquet",
    "scedc_socal_m25_features.parquet",
    "comcat_himalaya_m4_features.parquet",
]
REF_LABEL = "label_R50_T30"
FEATURES = ["mw", "depth", "gap", "rms", "dmin", "magError",
            "local_count_7d", "log_energy_30d", "mag_diff_30d",
            "time_gap_local", "seismicity_z", "latitude", "longitude"]
SPLITS = (0.60, 0.10, 0.15, 0.15)        # train / val / calib / test
N_TRIALS = 40
SEED = 42

MODELS_DIR = Path("./models")
MODELS_DIR.mkdir(exist_ok=True)

# GPU auto-detect (CatBoost / XGBoost only; LightGBM stays CPU on Colab)
try:
    import subprocess
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    GPU = False
print(f"GPU available: {GPU}")


# ----------------------------------------------------------------------------
def chrono_split(n):
    b1 = int(n * SPLITS[0])
    b2 = int(n * (SPLITS[0] + SPLITS[1]))
    b3 = int(n * (SPLITS[0] + SPLITS[1] + SPLITS[2]))
    return b1, b2, b3

def load_catalog(path):
    df = pd.read_parquet(path).sort_values("time").reset_index(drop=True)
    feats = [f for f in FEATURES if f in df.columns]
    X = df[feats].astype(np.float32)
    y = df[REF_LABEL].to_numpy(dtype=np.int8)
    return df, X, y, feats

def impute(name, X, b1):
    """MICE fitted on train only, applied frozen everywhere."""
    from sklearn.experimental import enable_iterative_imputer  # noqa
    from sklearn.impute import IterativeImputer
    p = MODELS_DIR / f"{name}__imputer.joblib"
    if p.exists():
        imp = joblib.load(p)
    else:
        imp = IterativeImputer(max_iter=10, random_state=SEED,
                               sample_posterior=False)
        imp.fit(X.iloc[:b1])
        joblib.dump(imp, p)
    return pd.DataFrame(imp.transform(X), columns=X.columns).astype(np.float32)


# ----------------------------------------------------------------------------
def make_model(kind, params):
    if kind == "catboost":
        from catboost import CatBoostClassifier
        return CatBoostClassifier(
            **params, random_seed=SEED, verbose=0,
            task_type="GPU" if GPU else "CPU",
            devices="0" if GPU else None,
        )
    if kind == "xgboost":
        from xgboost import XGBClassifier
        return XGBClassifier(
            **params, random_state=SEED, n_jobs=-1,
            tree_method="hist", device="cuda" if GPU else "cpu",
            eval_metric="logloss",
        )
    if kind == "lightgbm":
        from lightgbm import LGBMClassifier
        return LGBMClassifier(**params, random_state=SEED, n_jobs=-1,
                              verbosity=-1)
    raise ValueError(kind)


def search_space(kind, trial):
    p = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3,
                                             log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
    }
    if kind == "catboost":
        p["depth"] = trial.suggest_int("depth", 4, 10)
        p["l2_leaf_reg"] = trial.suggest_float("l2_leaf_reg", 1.0, 10.0)
        p["iterations"] = p.pop("n_estimators")
    else:
        p["max_depth"] = trial.suggest_int("max_depth", 4, 10)
        p["subsample"] = trial.suggest_float("subsample", 0.6, 1.0)
        p["colsample_bytree"] = trial.suggest_float("colsample_bytree",
                                                    0.6, 1.0)
        p["reg_lambda"] = trial.suggest_float("reg_lambda", 1.0, 10.0)
        if kind == "lightgbm":
            p["num_leaves"] = trial.suggest_int("num_leaves", 31, 255)
    return p


def tune(name, kind, Xtr, ytr, Xva, yva):
    import optuna
    from sklearn.metrics import f1_score
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    best_path = MODELS_DIR / f"{name}__optuna_best.json"
    best = json.loads(best_path.read_text()) if best_path.exists() else {}
    if kind in best:
        return best[kind]

    def objective(trial):
        m = make_model(kind, search_space(kind, trial))
        m.fit(Xtr, ytr)
        return f1_score(yva, m.predict(Xva))

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    best[kind] = study.best_params
    best_path.write_text(json.dumps(best, indent=2))
    return study.best_params


# ----------------------------------------------------------------------------
def evaluate(y_true, prob, thr=0.5):
    from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                                 recall_score, accuracy_score, brier_score_loss)
    pred = (prob >= thr).astype(int)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "auc": roc_auc_score(y_true, prob),
        "brier": brier_score_loss(y_true, prob),
    }


def process(path, metrics_rows):
    name = Path(path).stem.replace("_features", "")
    print(f"\n===== {name} =====")
    df, X, y, feats = load_catalog(path)
    n = len(df)
    b1, b2, b3 = chrono_split(n)
    np.savez(MODELS_DIR / f"{name}__splits.npz", b1=b1, b2=b2, b3=b3, n=n)
    print(f"n={n} | train {b1} | val {b2-b1} | calib {b3-b2} | test {n-b3}")
    print(f"mainshock rate: train {y[:b1].mean():.3f} "
          f"test {y[b3:].mean():.3f}")

    Xi = impute(name, X, b1)
    Xtr, ytr = Xi.iloc[:b1], y[:b1]
    Xva, yva = Xi.iloc[b1:b2], y[b1:b2]
    Xte, yte = Xi.iloc[b3:], y[b3:]

    # heuristic baseline: mainshock iff bigger than everything recent
    heur = (df["mag_diff_30d"].to_numpy() >= 0).astype(float)
    row = {"catalog": name, "model": "heuristic_magdiff",
           **evaluate(yte, heur[b3:])}
    metrics_rows.append(row)
    print(pd.DataFrame([row]).round(3).to_string(index=False))

    for kind in ["catboost", "xgboost", "lightgbm"]:
        probs_path = MODELS_DIR / f"{name}__{kind}__probs.npz"
        if probs_path.exists():
            print(f"[skip] {kind} already trained")
            prob = np.load(probs_path)["prob"]
        else:
            params = tune(name, kind, Xtr, ytr, Xva, yva)
            m = make_model(kind, params)
            m.fit(Xtr, ytr)
            prob = m.predict_proba(Xi)[:, 1]
            np.savez(probs_path, prob=prob.astype(np.float32))
            if kind == "catboost":
                m.save_model(str(MODELS_DIR / f"{name}__catboost.cbm"))
            else:
                joblib.dump(m, MODELS_DIR / f"{name}__{kind}.joblib")
            del m
            gc.collect()
        row = {"catalog": name, "model": kind, **evaluate(yte, prob[b3:])}
        metrics_rows.append(row)
        print(pd.DataFrame([row]).round(3).to_string(index=False))

    del df, X, Xi, y
    gc.collect()


if __name__ == "__main__":
    rows = []
    for p in CATALOGS:
        process(p, rows)
    out = pd.DataFrame(rows).round(4)
    out.to_csv("step5_metrics.csv", index=False)
    print("\n=== step5_metrics.csv ===")
    print(out.to_string(index=False))

GPU available: False

===== comcat_global_m4 =====
n=541817 | train 325090 | val 54181 | calib 81273 | test 81273
mainshock rate: train 0.530 test 0.416
         catalog             model  accuracy  precision  recall    f1   auc  brier
comcat_global_m4 heuristic_magdiff     0.896        0.8     1.0 0.889 0.911  0.104
[skip] catboost already trained
         catalog    model  accuracy  precision  recall    f1   auc  brier
comcat_global_m4 catboost     0.903      0.823   0.978 0.894 0.955  0.072
[skip] xgboost already trained
         catalog   model  accuracy  precision  recall    f1   auc  brier
comcat_global_m4 xgboost     0.903      0.825   0.975 0.894 0.955  0.071
[skip] lightgbm already trained
         catalog    model  accuracy  precision  recall    f1   auc  brier
comcat_global_m4 lightgbm     0.904      0.828   0.972 0.894 0.955  0.071

===== scedc_socal_m25 =====
n=81109 | train 48665 | val 8111 | calib 12166 | test 12167
mainshock rate: train 0.255 test 0.325
        catalog 

           catalog    model  accuracy  precision  recall    f1   auc  brier
comcat_himalaya_m4 lightgbm     0.923      0.886     1.0 0.939 0.938  0.066

=== step5_metrics.csv ===
           catalog             model  accuracy  precision  recall     f1    auc  brier
  comcat_global_m4 heuristic_magdiff    0.8959     0.7998  1.0000 0.8888 0.9108 0.1041
  comcat_global_m4          catboost    0.9033     0.8232  0.9776 0.8938 0.9545 0.0716
  comcat_global_m4           xgboost    0.9035     0.8248  0.9751 0.8937 0.9547 0.0714
  comcat_global_m4          lightgbm    0.9041     0.8276  0.9718 0.8939 0.9546 0.0714
   scedc_socal_m25 heuristic_magdiff    0.9188     0.8002  1.0000 0.8890 0.9398 0.0812
   scedc_socal_m25          catboost    0.9356     0.8591  0.9590 0.9064 0.9785 0.0481
   scedc_socal_m25           xgboost    0.9358     0.8602  0.9583 0.9066 0.9784 0.0479
   scedc_socal_m25          lightgbm    0.9352     0.8574  0.9603 0.9059 0.9785 0.0478
comcat_himalaya_m4 heuristic_magdiff  

In [ ]:

import json
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
CATALOGS = ["comcat_global_m4", "scedc_socal_m25", "comcat_himalaya_m4"]
MODELS = ["catboost", "xgboost", "lightgbm"]
REF_LABEL = "label_R50_T30"
ALPHAS = [0.05, 0.10, 0.20]
ACI_GAMMA = 0.005
N_BINS = 15
MODELS_DIR = Path("./models")


# ----------------------------------------------------------------------------
# Calibration metrics + recalibrators
# ----------------------------------------------------------------------------
def ece(y, p, n_bins=N_BINS):
    """Expected calibration error with equal-frequency bins."""
    order = np.argsort(p)
    y, p = y[order], p[order]
    bins = np.array_split(np.arange(len(p)), n_bins)
    e = 0.0
    for b in bins:
        if len(b) == 0:
            continue
        e += (len(b) / len(p)) * abs(y[b].mean() - p[b].mean())
    return e


def nll(y, p, eps=1e-7):
    p = np.clip(p, eps, 1 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def reliability_points(y, p, n_bins=N_BINS):
    order = np.argsort(p)
    y, p = y[order], p[order]
    rows = []
    for b in np.array_split(np.arange(len(p)), n_bins):
        if len(b):
            rows.append({"mean_pred": p[b].mean(), "frac_pos": y[b].mean(),
                         "count": len(b)})
    return rows


def fit_recalibrators(p_val, y_val):
    from sklearn.linear_model import LogisticRegression
    from sklearn.isotonic import IsotonicRegression
    from scipy.optimize import minimize_scalar

    out = {}
    # Platt: logistic regression on the logit
    z = np.log(np.clip(p_val, 1e-7, 1 - 1e-7) /
               np.clip(1 - p_val, 1e-7, 1 - 1e-7)).reshape(-1, 1)
    lr = LogisticRegression(C=1e6)
    lr.fit(z, y_val)
    out["platt"] = ("platt", lr)
    # Isotonic
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
    iso.fit(p_val, y_val)
    out["isotonic"] = ("isotonic", iso)
    # Temperature scaling on logits: minimize val NLL over T
    zv = z.ravel()

    def nll_at_T(T):
        pT = 1.0 / (1.0 + np.exp(-zv / max(T, 1e-3)))
        return nll(y_val, pT)

    res = minimize_scalar(nll_at_T, bounds=(0.05, 20.0), method="bounded")
    out["temperature"] = ("temperature", float(res.x))
    return out


def apply_recal(kind_obj, p):
    kind, obj = kind_obj
    z = np.log(np.clip(p, 1e-7, 1 - 1e-7) / np.clip(1 - p, 1e-7, 1 - 1e-7))
    if kind == "platt":
        return obj.predict_proba(z.reshape(-1, 1))[:, 1]
    if kind == "isotonic":
        return obj.predict(p)
    if kind == "temperature":
        return 1.0 / (1.0 + np.exp(-z / obj))
    raise ValueError(kind)


# ----------------------------------------------------------------------------
# Conformal machinery (binary; score = 1 - p_true_class)
# ----------------------------------------------------------------------------
def split_threshold(scores_cal, alpha):
    n = len(scores_cal)
    q = np.ceil((n + 1) * (1 - alpha)) / n
    return np.quantile(scores_cal, min(q, 1.0), method="higher")


def prediction_sets(p, qhat):
    """Return boolean arrays (in_set_class0, in_set_class1)."""
    s1 = 1.0 - p           # nonconformity if true class were 1
    s0 = p                 # nonconformity if true class were 0
    return s0 <= qhat, s1 <= qhat


def set_metrics(y, in0, in1):
    size = in0.astype(int) + in1.astype(int)
    covered = np.where(y == 1, in1, in0)
    return {
        "coverage": float(covered.mean()),
        "coverage_class1": float(covered[y == 1].mean()) if (y == 1).any() else np.nan,
        "coverage_class0": float(covered[y == 0].mean()) if (y == 0).any() else np.nan,
        "avg_set_size": float(size.mean()),
        "singleton_rate": float((size == 1).mean()),
        "abstain_rate": float((size == 2).mean()),
        "empty_rate": float((size == 0).mean()),
    }


def mondrian_thresholds(p_cal, y_cal, alpha):
    q1 = split_threshold(1.0 - p_cal[y_cal == 1], alpha)
    q0 = split_threshold(p_cal[y_cal == 0], alpha)
    return q0, q1


def mondrian_sets(p, q0, q1):
    return p <= q0, (1.0 - p) <= q1


def aci(p_test, y_test, p_cal, y_cal, alpha, gamma=ACI_GAMMA):
    """Adaptive Conformal Inference over the chronological test block.
    Recomputes the split threshold at the adapted level each step using
    the fixed calibration scores. Returns per-step coverage + set size."""
    scores_cal_1 = 1.0 - p_cal[y_cal == 1]
    scores_cal_0 = p_cal[y_cal == 0]
    cal_scores = np.where(y_cal == 1, 1.0 - p_cal, p_cal)
    a_t = alpha
    cov, size = np.zeros(len(y_test)), np.zeros(len(y_test))
    for t in range(len(y_test)):
        a_eff = float(np.clip(a_t, 1e-4, 0.999))
        qhat = split_threshold(cal_scores, a_eff)
        in0, in1 = p_test[t] <= qhat, (1.0 - p_test[t]) <= qhat
        covered = in1 if y_test[t] == 1 else in0
        cov[t] = covered
        size[t] = int(in0) + int(in1)
        a_t = a_t + gamma * (alpha - (0.0 if covered else 1.0))
    return cov, size


def rolling(x, w=2000):
    s = pd.Series(x)
    return s.rolling(w, min_periods=200).mean().to_numpy()


# ----------------------------------------------------------------------------
def main():
    cal_rows, conf_rows, rel_rows, aci_rows = [], [], [], []

    for cat in CATALOGS:
        spl = np.load(MODELS_DIR / f"{cat}__splits.npz")
        b1, b2, b3, n = int(spl["b1"]), int(spl["b2"]), int(spl["b3"]), int(spl["n"])
        y = pd.read_parquet(f"{cat}_features.parquet",
                            columns=[REF_LABEL])[REF_LABEL].to_numpy(np.int8)

        for model in MODELS:
            probs_path = MODELS_DIR / f"{cat}__{model}__probs.npz"
            if not probs_path.exists():
                print(f"[missing] {probs_path} — run step 5 first")
                continue
            p = np.load(probs_path)["prob"].astype(np.float64)
            p_val, y_val = p[b1:b2], y[b1:b2]
            p_cal, y_cal = p[b2:b3], y[b2:b3]
            p_te, y_te = p[b3:], y[b3:]

            # ---------- A) calibration audit ----------
            recals = fit_recalibrators(p_val, y_val)
            variants = {"raw": p_te}
            for rname, ko in recals.items():
                variants[rname] = apply_recal(ko, p_te)

            best_name, best_ece = "raw", np.inf
            for vname, pv in variants.items():
                row = {"catalog": cat, "model": model, "variant": vname,
                       "ece": ece(y_te, pv),
                       "brier": float(np.mean((pv - y_te) ** 2)),
                       "nll": nll(y_te, pv)}
                cal_rows.append(row)
                if vname != "raw" and row["ece"] < best_ece:
                    best_name, best_ece = vname, row["ece"]
                for pt in reliability_points(y_te, pv):
                    rel_rows.append({"catalog": cat, "model": model,
                                     "variant": vname, **pt})

            # recalibrated probabilities (full vector) for later steps
            best_ko = recals[best_name]
            p_recal = apply_recal(best_ko, p)
            np.savez(MODELS_DIR / f"{cat}__{model}__recal_probs.npz",
                     prob=p_recal.astype(np.float32),
                     method=np.array(best_name))

            # ---------- B/C) split + Mondrian conformal ----------
            for score_name, pv_full in [("raw", p), ("recal", p_recal)]:
                pc, pt = pv_full[b2:b3], pv_full[b3:]
                cal_scores = np.where(y_cal == 1, 1 - pc, pc)
                for alpha in ALPHAS:
                    qhat = split_threshold(cal_scores, alpha)
                    in0, in1 = prediction_sets(pt, qhat)
                    conf_rows.append({"catalog": cat, "model": model,
                                      "scores": score_name, "method": "split",
                                      "alpha": alpha,
                                      **set_metrics(y_te, in0, in1)})
                    q0, q1 = mondrian_thresholds(pc, y_cal, alpha)
                    in0m, in1m = mondrian_sets(pt, q0, q1)
                    conf_rows.append({"catalog": cat, "model": model,
                                      "scores": score_name,
                                      "method": "mondrian", "alpha": alpha,
                                      **set_metrics(y_te, in0m, in1m)})

            # ---------- D) ACI (recalibrated scores, alpha = 0.10) ----------
            cov_aci, size_aci = aci(p_recal[b3:], y_te,
                                    p_recal[b2:b3], y_cal, alpha=0.10)
            qfix = split_threshold(
                np.where(y_cal == 1, 1 - p_recal[b2:b3], p_recal[b2:b3]), 0.10)
            in0f, in1f = prediction_sets(p_recal[b3:], qfix)
            cov_fix = np.where(y_te == 1, in1f, in0f).astype(float)
            r_aci, r_fix = rolling(cov_aci), rolling(cov_fix)
            step = max(1, len(y_te) // 400)
            for t in range(0, len(y_te), step):
                aci_rows.append({"catalog": cat, "model": model, "t": t,
                                 "rolling_cov_split": r_fix[t],
                                 "rolling_cov_aci": r_aci[t]})
            print(f"{cat} / {model}: best recal = {best_name} "
                  f"(ECE {best_ece:.4f} vs raw "
                  f"{[r['ece'] for r in cal_rows if r['catalog']==cat and r['model']==model and r['variant']=='raw'][0]:.4f}) | "
                  f"ACI mean cov {cov_aci.mean():.3f} vs split {cov_fix.mean():.3f}")

    pd.DataFrame(cal_rows).round(5).to_csv("step6_calibration.csv", index=False)
    pd.DataFrame(conf_rows).round(4).to_csv("step6_conformal.csv", index=False)
    pd.DataFrame(rel_rows).round(5).to_csv("step6_reliability_data.csv", index=False)
    pd.DataFrame(aci_rows).round(4).to_csv("step6_aci_rolling.csv", index=False)

    print("\n=== calibration (test block) ===")
    print(pd.DataFrame(cal_rows).round(4).to_string(index=False))
    print("\n=== conformal (alpha = 0.10, recalibrated scores) ===")
    cdf = pd.DataFrame(conf_rows)
    print(cdf[(cdf.alpha == 0.10) & (cdf.scores == "recal")]
          .round(3).to_string(index=False))


if __name__ == "__main__":
    main()

comcat_global_m4 / catboost: best recal = isotonic (ECE 0.0057 vs raw 0.0100) | ACI mean cov 0.900 vs split 0.906
comcat_global_m4 / xgboost: best recal = temperature (ECE 0.0045 vs raw 0.0099) | ACI mean cov 0.900 vs split 0.905
comcat_global_m4 / lightgbm: best recal = temperature (ECE 0.0052 vs raw 0.0119) | ACI mean cov 0.900 vs split 0.905
scedc_socal_m25 / catboost: best recal = isotonic (ECE 0.0066 vs raw 0.0083) | ACI mean cov 0.899 vs split 0.886
scedc_socal_m25 / xgboost: best recal = isotonic (ECE 0.0082 vs raw 0.0097) | ACI mean cov 0.899 vs split 0.887
scedc_socal_m25 / lightgbm: best recal = isotonic (ECE 0.0066 vs raw 0.0114) | ACI mean cov 0.899 vs split 0.886
comcat_himalaya_m4 / catboost: best recal = isotonic (ECE 0.0236 vs raw 0.0514) | ACI mean cov 0.920 vs split 0.920
comcat_himalaya_m4 / xgboost: best recal = platt (ECE 0.0173 vs raw 0.0229) | ACI mean cov 0.896 vs split 0.886
comcat_himalaya_m4 / lightgbm: best recal = isotonic (ECE 0.0284 vs raw 0.0344) | ACI m

In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
CATALOGS = ["comcat_global_m4", "scedc_socal_m25", "comcat_himalaya_m4"]
RADII_KM = [25, 50, 75, 100]
WINDOWS_DAYS = [15, 30, 60, 90]
REF_LABEL = "label_R50_T30"
FEATURES = ["mw", "depth", "gap", "rms", "dmin", "magError",
            "local_count_7d", "log_energy_30d", "mag_diff_30d",
            "time_gap_local", "seismicity_z", "latitude", "longitude"]
ALPHA = 0.10
SEED = 42

MODELS_DIR = Path("./models")
GRID_CKPT = Path("./grid_ckpt")
GRID_CKPT.mkdir(exist_ok=True)

try:
    import subprocess
    GPU = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except Exception:
    GPU = False
print(f"GPU available: {GPU}")


def all_label_cols():
    cols = [f"label_R{R}_T{T}" for R in RADII_KM for T in WINDOWS_DAYS]
    cols.append("label_adaptive")
    return cols


# ---------- reuse step-6 machinery (self-contained copies) ----------
def ece(y, p, n_bins=15):
    order = np.argsort(p)
    y, p = y[order], p[order]
    e = 0.0
    for b in np.array_split(np.arange(len(p)), n_bins):
        if len(b):
            e += (len(b) / len(p)) * abs(y[b].mean() - p[b].mean())
    return e


def split_threshold(scores_cal, alpha):
    n = len(scores_cal)
    q = np.ceil((n + 1) * (1 - alpha)) / n
    return np.quantile(scores_cal, min(q, 1.0), method="higher")


def mondrian_metrics(p_cal, y_cal, p_te, y_te, alpha=ALPHA):
    q1 = split_threshold(1.0 - p_cal[y_cal == 1], alpha)
    q0 = split_threshold(p_cal[y_cal == 0], alpha)
    in0, in1 = p_te <= q0, (1.0 - p_te) <= q1
    size = in0.astype(int) + in1.astype(int)
    covered = np.where(y_te == 1, in1, in0)
    return {
        "cov_class1": float(covered[y_te == 1].mean()) if (y_te == 1).any() else np.nan,
        "cov_class0": float(covered[y_te == 0].mean()) if (y_te == 0).any() else np.nan,
        "avg_set_size": float(size.mean()),
        "abstain_rate": float((size == 2).mean()),
    }


def isotonic_recal(p_val, y_val, p_te):
    from sklearn.isotonic import IsotonicRegression
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
    iso.fit(p_val, y_val)
    return iso.predict(p_te)


def kendall_tau(rank_a, rank_b):
    """Kendall tau between two orderings given as lists of feature names."""
    from scipy.stats import kendalltau
    common = [f for f in rank_a if f in rank_b]
    ra = [rank_a.index(f) for f in common]
    rb = [rank_b.index(f) for f in common]
    return float(kendalltau(ra, rb).statistic)


# ----------------------------------------------------------------------------
def run_config(cat, label_col, X, labels, feats, b1, b2, b3, params):
    """Train + evaluate one (catalog, label config). Returns result row
    and SHAP ranking. Checkpointed."""
    ck = GRID_CKPT / f"{cat}__{label_col}.json"
    if ck.exists():
        return json.loads(ck.read_text())

    from xgboost import XGBClassifier
    from sklearn.metrics import roc_auc_score, f1_score, recall_score, \
        accuracy_score

    y = labels[label_col].to_numpy(dtype=np.int8)
    ytr, yva, yca, yte = y[:b1], y[b1:b2], y[b2:b3], y[b3:]

    m = XGBClassifier(**params, random_state=SEED, n_jobs=-1,
                      tree_method="hist",
                      device="cuda" if GPU else "cpu",
                      eval_metric="logloss")
    m.fit(X.iloc[:b1], ytr)
    prob = m.predict_proba(X)[:, 1].astype(np.float64)
    p_va, p_ca, p_te = prob[b1:b2], prob[b2:b3], prob[b3:]

    pred = (p_te >= 0.5).astype(int)
    p_te_recal = isotonic_recal(p_va, yva, p_te)
    p_ca_recal = isotonic_recal(p_va, yva, p_ca)

    # GPU TreeSHAP global importances on a test subsample
    import xgboost as xgb
    sub = X.iloc[b3:].iloc[:20000]
    booster = m.get_booster()
    dm = xgb.DMatrix(sub)
    contribs = booster.predict(dm, pred_contribs=True)   # (n, n_feat+1)
    mean_abs = np.abs(contribs[:, :-1]).mean(axis=0)
    ranking = [feats[i] for i in np.argsort(-mean_abs)]

    row = {
        "catalog": cat, "config": label_col,
        "mainshock_rate": float(y.mean()),
        "auc": float(roc_auc_score(yte, p_te)),
        "f1": float(f1_score(yte, pred)),
        "recall": float(recall_score(yte, pred)),
        "accuracy": float(accuracy_score(yte, pred)),
        "ece_recal": float(ece(yte, p_te_recal)),
        "brier": float(np.mean((p_te - yte) ** 2)),
        **mondrian_metrics(p_ca_recal, yca, p_te_recal, yte),
        "shap_ranking": ranking,
        "shap_top1": ranking[0],
    }
    ck.write_text(json.dumps(row))
    del m, prob
    gc.collect()
    return row


def main():
    rows, shap_rows = [], []
    for cat in CATALOGS:
        print(f"\n===== {cat} =====")
        spl = np.load(MODELS_DIR / f"{cat}__splits.npz")
        b1, b2, b3 = int(spl["b1"]), int(spl["b2"]), int(spl["b3"])

        df = pd.read_parquet(f"{cat}_features.parquet")
        df = df.sort_values("time").reset_index(drop=True)
        feats = [f for f in FEATURES if f in df.columns]
        X_raw = df[feats].astype(np.float32)
        labels = df[all_label_cols()]
        ref_y = labels[REF_LABEL].to_numpy(dtype=np.int8)

        # frozen imputer + frozen hyperparameters from step 5
        import joblib
        imp = joblib.load(MODELS_DIR / f"{cat}__imputer.joblib")
        X = pd.DataFrame(imp.transform(X_raw), columns=feats)\
              .astype(np.float32)
        best = json.loads((MODELS_DIR / f"{cat}__optuna_best.json")
                          .read_text())["xgboost"]

        cfg_rows = []
        for col in all_label_cols():
            r = run_config(cat, col, X, labels, feats, b1, b2, b3, best)
            r["flip_rate_vs_ref"] = float(
                (labels[col].to_numpy(dtype=np.int8) != ref_y).mean())
            cfg_rows.append(r)
            print(f"  {col}: AUC {r['auc']:.3f} F1 {r['f1']:.3f} "
                  f"flip {r['flip_rate_vs_ref']:.3f} "
                  f"abstain {r['abstain_rate']:.3f} top1 {r['shap_top1']}")

        # SHAP stability vs reference
        ref_rank = next(r for r in cfg_rows
                        if r["config"] == REF_LABEL)["shap_ranking"]
        for r in cfg_rows:
            r["shap_kendall_tau_vs_ref"] = kendall_tau(r["shap_ranking"],
                                                       ref_rank)
            r["shap_top5_overlap"] = len(set(r["shap_ranking"][:5])
                                         & set(ref_rank[:5])) / 5.0
            shap_rows.append({"catalog": cat, "config": r["config"],
                              "ranking": " > ".join(r["shap_ranking"][:6])})
            rows.append({k: v for k, v in r.items() if k != "shap_ranking"})

        del df, X, X_raw, labels
        gc.collect()

    out = pd.DataFrame(rows).round(4)
    out.to_csv("step7_sensitivity_grid.csv", index=False)
    pd.DataFrame(shap_rows).to_csv("step7_shap_rankings.csv", index=False)

    print("\n=== step7_sensitivity_grid.csv ===")
    print(out.to_string(index=False))

    # H1 headline: spread of metrics vs spread of labels
    print("\n=== H1 summary (per catalog) ===")
    for cat in CATALOGS:
        s = out[out.catalog == cat]
        print(f"{cat}: AUC range {s.auc.min():.3f}-{s.auc.max():.3f} | "
              f"F1 range {s.f1.min():.3f}-{s.f1.max():.3f} | "
              f"flip up to {s.flip_rate_vs_ref.max():.1%} | "
              f"tau min {s.shap_kendall_tau_vs_ref.min():.2f}")


if __name__ == "__main__":
    main()

GPU available: False

===== comcat_global_m4 =====
  label_R25_T15: AUC 0.931 F1 0.894 flip 0.186 abstain 0.109 top1 mag_diff_30d
  label_R25_T30: AUC 0.928 F1 0.871 flip 0.132 abstain 0.111 top1 mag_diff_30d
  label_R25_T60: AUC 0.916 F1 0.834 flip 0.139 abstain 0.185 top1 mag_diff_30d
  label_R25_T90: AUC 0.912 F1 0.806 flip 0.149 abstain 0.189 top1 mag_diff_30d
  label_R50_T15: AUC 0.948 F1 0.906 flip 0.078 abstain 0.076 top1 mag_diff_30d
  label_R50_T30: AUC 0.955 F1 0.894 flip 0.000 abstain 0.042 top1 mag_diff_30d
  label_R50_T60: AUC 0.950 F1 0.822 flip 0.087 abstain 0.053 top1 mag_diff_30d
  label_R50_T90: AUC 0.948 F1 0.778 flip 0.139 abstain 0.056 top1 mag_diff_30d
  label_R75_T15: AUC 0.930 F1 0.842 flip 0.107 abstain 0.146 top1 mag_diff_30d
  label_R75_T30: AUC 0.939 F1 0.803 flip 0.085 abstain 0.095 top1 mag_diff_30d
  label_R75_T60: AUC 0.941 F1 0.727 flip 0.175 abstain 0.088 top1 mag_diff_30d
  label_R75_T90: AUC 0.942 F1 0.678 flip 0.224 abstain 0.076 top1 mag_diff_30d
 

In [ ]:
import nbformat

nb = nbformat.read("notebook.ipynb", as_version=4)

widgets = nb.metadata.get("widgets")

if widgets:
    for key, value in widgets.items():
        if isinstance(value, dict) and "state" not in value:
            value["state"] = {}

nbformat.write(nb, "notebook_fixed.ipynb")
print("Fixed notebook saved.")

FileNotFoundError: [Errno 2] No such file or directory: 'notebook.ipynb'